In [2]:
import numpy as np 
a = [1, 2, 3, 4]
feature_vector_sorted = a

In [5]:
threshholds  = np.lib.stride_tricks.sliding_window_view(feature_vector_sorted, 2).mean(axis=1)


In [8]:
np.zeros(a)

array([[[[0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.]],

        [[0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.]]]])

In [ ]:

def find_best_split(feature_vector, target_vector):
    """
    Под критерием Джини здесь подразумевается следующая функция:
    $$Q(R) = -\frac {|R_l|}{|R|}H(R_l) -\frac {|R_r|}{|R|}H(R_r)$$,
    $R$ — множество объектов, $R_l$ и $R_r$ — объекты, попавшие в левое и правое поддерево,
     $H(R) = 1-p_1^2-p_0^2$, $p_1$, $p_0$ — доля объектов класса 1 и 0 соответственно.

    Указания:
    * Пороги, приводящие к попаданию в одно из поддеревьев пустого множества объектов, не рассматриваются.
    * В качестве порогов, нужно брать среднее двух сосдених (при сортировке) значений признака
    * Поведение функции в случае константного признака может быть любым.
    * При одинаковых приростах Джини нужно выбирать минимальный сплит.
    * За наличие в функции циклов балл будет снижен. Векторизуйте! :)

    :param feature_vector: вещественнозначный вектор значений признака
    :param target_vector: вектор классов объектов,  len(feature_vector) == len(target_vector)

    :return thresholds: отсортированный по возрастанию вектор со всеми возможными порогами, по которым объекты можно
     разделить на две различные подвыборки, или поддерева
    :return ginis: вектор со значениями критерия Джини для каждого из порогов в thresholds len(ginis) == len(thresholds)
    :return threshold_best: оптимальный порог (число)
    :return gini_best: оптимальное значение критерия Джини (число)
    """

    dict_results = {}   # Храним результаты
   

    #Функция для расчета критерия gini

    def gini(left_split: np.array, right_split: np.array):
        
        p1,p0 = 0,0 
        size_left, size_right = np.size(left_split), np.size(right_split)
        size_full = size_left + size_right

        if size_left == 0 or size_right == 0:
            print('Пустое поддерево')
            return np.inf

        #Расчет H(left_split) для левого поддерева 
        non_zero = np.count_nonzero(left_split)
        zeros = size_left - non_zero 

        p1 = non_zero/size_left 
        p0 = zeros/size_left 

        H_left = 1 - p1**2 - p0**2

        #Расчет H(right_split) для правого поддерева
        non_zero = np.count_nonzero(right_split)
        zeros = size_right - non_zero 

        p1 = non_zero/size_right 
        p0 = zeros/size_right 

        H_right = 1 - p1**2 - p0**2


        return - size_left/size_full*H_left - size_right/size_full*H_right 


    #Расчет трешхолдов 
    feature_vector_sort = sorted(feature_vector)
    thresholds = np.lib.stride_tricks.sliding_window_view(feature_vector_sort, 2).mean(axis=1) # Через скользящее


    dict_results = {}
    for thresh in thresholds:
        mask_left = feature_vector <= thresh
        mask_right = feature_vector > thresh
        left_split = target_vector[mask_left]
        right_split = target_vector[mask_right]
        g = gini(left_split, right_split)
        dict_results[thresh] = (left_split, right_split, g)
        ginis.append(g)

    ginis = np.array(ginis)
    idx_best = np.argmin(ginis)
    threshold_best = thresholds[idx_best]
    gini_best = ginis[idx_best]

    return thresholds, ginis, threshold_best, gini_best